In [3]:
import time

import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
import torch_geometric.transforms as T
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import VGAE, GCNConv
from sklearn.metrics import roc_auc_score, average_precision_score


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# -----------------------------
# Encoder
# -----------------------------
class Encoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()

        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv_mu = GCNConv(hidden_channels, out_channels)
        self.conv_logstd = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)

        mu = self.conv_mu(x, edge_index)
        logstd = self.conv_logstd(x, edge_index)
        return mu, logstd

# -----------------------------
# Train
# -----------------------------
def train(model, train_data, optimizer):
    model.train()
    optimizer.zero_grad()

    z = model.encode(
        train_data.x,
        train_data.edge_index,
    )

    loss = model.recon_loss(
        z,
        train_data.edge_label_index,
    )
    loss = loss +  (1 / train_data.num_nodes) * model.kl_loss()

    loss.backward()
    optimizer.step()

    return loss.item()

# -----------------------------
# Evaluation
# -----------------------------
@torch.no_grad()
def test(model, data):
    model.eval()

    z = model.encode(
        data.x,
        data.edge_index,
    )

    pred = model.decoder(
        z,
        data.edge_label_index,
        sigmoid=True
    )

    y = data.edge_label.cpu().numpy()
    pred = pred.cpu().numpy()

    auc = roc_auc_score(y, pred)
    ap = average_precision_score(y, pred)

    return auc, ap



In [9]:
# - Cora Dataset test---
dataset = Planetoid(root='./data/Cora', name="Cora")
data = dataset[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")

# 데이터 불러오기/전처리
data = dataset[0]
transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(data)
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

# 모델 생성
encoder = Encoder(
    dataset.num_features,
    hidden_channels=64,
    out_channels=32,
)

model = VGAE(encoder).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)
# -----------------------------
# Training Loop
# -----------------------------
best_val_auc = 0

print(f'---dataset is Cora----')
print(f'---device is {device} ---')
print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 201):

    loss = train(model=model, train_data=train_data, optimizer=optimizer)

    val_auc, val_ap = test(model=model, data=val_data)
        
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Val AUC {val_auc:.4f} | "
            f"Val AP {val_ap:.4f}"
        )

print(f"--------학습 완료--------")

test_auc, test_ap = test(model=model, data=test_data)
print(f"--------Cora: Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time:.4f}")


c:\Users\pmw9440\AppData\Local\pypoetry\Cache\virtualenvs\gae-y3S4qk4U-py3.11\Lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental featu

---dataset is Cora----
---device is cuda ---
------학습시작-------
Epoch 010 | Loss 1.4054 | Val AUC 0.8096 | Val AP 0.8207
Epoch 020 | Loss 1.2805 | Val AUC 0.9016 | Val AP 0.9070
Epoch 030 | Loss 1.2168 | Val AUC 0.9135 | Val AP 0.9178
Epoch 040 | Loss 1.1724 | Val AUC 0.9186 | Val AP 0.9274
Epoch 050 | Loss 1.1427 | Val AUC 0.9118 | Val AP 0.9245
Epoch 060 | Loss 1.1164 | Val AUC 0.9072 | Val AP 0.9211
Epoch 070 | Loss 1.0996 | Val AUC 0.8993 | Val AP 0.9150
Epoch 080 | Loss 1.0849 | Val AUC 0.8988 | Val AP 0.9143
Epoch 090 | Loss 1.0793 | Val AUC 0.8973 | Val AP 0.9136
Epoch 100 | Loss 1.0713 | Val AUC 0.8911 | Val AP 0.9086
Epoch 110 | Loss 1.0599 | Val AUC 0.8873 | Val AP 0.9053
Epoch 120 | Loss 1.0578 | Val AUC 0.8909 | Val AP 0.9077
Epoch 130 | Loss 1.0549 | Val AUC 0.8883 | Val AP 0.9065
Epoch 140 | Loss 1.0478 | Val AUC 0.8894 | Val AP 0.9112
Epoch 150 | Loss 1.0373 | Val AUC 0.8834 | Val AP 0.9072
Epoch 160 | Loss 1.0338 | Val AUC 0.8770 | Val AP 0.9015
Epoch 170 | Loss 1.0410 |

In [10]:
# - Citeseer Dataset test---
dataset = Planetoid(root='./data/Citeseer', name="Citeseer")
data = dataset[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

# 데이터 불러오기/전처리
data = dataset[0]
transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(data)
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

# 모델 생성
encoder = Encoder(
    dataset.num_features,
    hidden_channels=64,
    out_channels=32,
)

model = VGAE(encoder).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)
# -----------------------------
# Training Loop
# -----------------------------
best_val_auc = 0

print(f'---dataset is Citeseer----')
print(f'---device is {device} ---')
print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 201):

    loss = train(model=model, train_data=train_data, optimizer=optimizer)

    val_auc, val_ap = test(model=model, data=val_data)
        
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Val AUC {val_auc:.4f} | "
            f"Val AP {val_ap:.4f}"
        )

print(f"--------학습 완료--------")

test_auc, test_ap = test(model=model, data=test_data)
print(f"--------Citeseer: Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time:.4f}")


c:\Users\pmw9440\AppData\Local\pypoetry\Cache\virtualenvs\gae-y3S4qk4U-py3.11\Lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental featu

---dataset is Citeseer----
---device is cpu ---
------학습시작-------
Epoch 010 | Loss 1.3339 | Val AUC 0.8644 | Val AP 0.8829
Epoch 020 | Loss 1.2082 | Val AUC 0.8854 | Val AP 0.9039
Epoch 030 | Loss 1.1118 | Val AUC 0.8803 | Val AP 0.8992
Epoch 040 | Loss 1.0509 | Val AUC 0.8797 | Val AP 0.8990
Epoch 050 | Loss 1.0157 | Val AUC 0.8763 | Val AP 0.9002
Epoch 060 | Loss 0.9953 | Val AUC 0.8737 | Val AP 0.8985
Epoch 070 | Loss 0.9790 | Val AUC 0.8779 | Val AP 0.9030
Epoch 080 | Loss 0.9802 | Val AUC 0.8736 | Val AP 0.8991
Epoch 090 | Loss 0.9749 | Val AUC 0.8719 | Val AP 0.8973
Epoch 100 | Loss 0.9716 | Val AUC 0.8691 | Val AP 0.8951
Epoch 110 | Loss 0.9664 | Val AUC 0.8683 | Val AP 0.8927
Epoch 120 | Loss 0.9453 | Val AUC 0.8657 | Val AP 0.8901
Epoch 130 | Loss 0.9564 | Val AUC 0.8677 | Val AP 0.8879
Epoch 140 | Loss 0.9521 | Val AUC 0.8686 | Val AP 0.8884
Epoch 150 | Loss 0.9623 | Val AUC 0.8650 | Val AP 0.8899
Epoch 160 | Loss 0.9508 | Val AUC 0.8568 | Val AP 0.8867
Epoch 170 | Loss 0.940

In [11]:
# - Pubmed Dataset test---
dataset = Planetoid(root='./data/Pubmed', name="Pubmed")
data = dataset[0]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")

# 데이터 불러오기/전처리
data = dataset[0]
transform = RandomLinkSplit(num_val=0.1, num_test=0.1, is_undirected=True, add_negative_train_samples=True)
train_data, val_data, test_data = transform(data)
train_data = train_data.to(device)
val_data = val_data.to(device)
test_data = test_data.to(device)

# 모델 생성
encoder = Encoder(
    dataset.num_features,
    hidden_channels=64,
    out_channels=32,
)

model = VGAE(encoder).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
    weight_decay=5e-4,
)
# -----------------------------
# Training Loop
# -----------------------------
best_val_auc = 0

print(f'---dataset is Pubmed----')
print(f'---device is {device} ---')
print('------학습시작-------')
start_time = time.time()
for epoch in range(1, 201):

    loss = train(model=model, train_data=train_data, optimizer=optimizer)

    val_auc, val_ap = test(model=model, data=val_data)
        
    if epoch % 10 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss {loss:.4f} | "
            f"Val AUC {val_auc:.4f} | "
            f"Val AP {val_ap:.4f}"
        )

print(f"--------학습 완료--------")

test_auc, test_ap = test(model=model, data=test_data)
print(f"--------Pubmed: Test AUC: {test_auc:.4f}, Test AP: {test_ap:.4f}")
print(f"--------총 걸린시간(s): {time.time() - start_time:.4f}")


c:\Users\pmw9440\AppData\Local\pypoetry\Cache\virtualenvs\gae-y3S4qk4U-py3.11\Lib\site-packages\torch_geometric\data\dataset.py:238: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental featu

---dataset is Pubmed----
---device is cpu ---
------학습시작-------
Epoch 010 | Loss 1.6480 | Val AUC 0.8809 | Val AP 0.8592
Epoch 020 | Loss 1.3516 | Val AUC 0.8807 | Val AP 0.8595
Epoch 030 | Loss 1.3369 | Val AUC 0.8802 | Val AP 0.8616
Epoch 040 | Loss 1.3262 | Val AUC 0.8724 | Val AP 0.8601
Epoch 050 | Loss 1.3190 | Val AUC 0.8671 | Val AP 0.8614
Epoch 060 | Loss 1.3070 | Val AUC 0.8912 | Val AP 0.8832
Epoch 070 | Loss 1.2835 | Val AUC 0.8945 | Val AP 0.8903
Epoch 080 | Loss 1.2773 | Val AUC 0.9012 | Val AP 0.8956
Epoch 090 | Loss 1.2763 | Val AUC 0.8999 | Val AP 0.8952
Epoch 100 | Loss 1.2729 | Val AUC 0.9026 | Val AP 0.8980
Epoch 110 | Loss 1.2704 | Val AUC 0.9034 | Val AP 0.9001
Epoch 120 | Loss 1.2658 | Val AUC 0.9087 | Val AP 0.9059
Epoch 130 | Loss 1.2508 | Val AUC 0.9147 | Val AP 0.9133
Epoch 140 | Loss 1.2451 | Val AUC 0.9129 | Val AP 0.9131
Epoch 150 | Loss 1.2441 | Val AUC 0.9158 | Val AP 0.9145
Epoch 160 | Loss 1.2422 | Val AUC 0.9137 | Val AP 0.9132
Epoch 170 | Loss 1.2392 